<a href="https://colab.research.google.com/github/LCaravaggio/FelicidadDesigualdad/blob/main/Train_Test_OCDE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import userdata
import json

!mkdir ~/.kaggle
!touch ~/.kaggle/kaggle.json

api_token = {
    'username': userdata.get('KAGGLE_USER'),
    'key': userdata.get('KAGGLE_KEY')}
with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(api_token, file)

!chmod 600 ~/.kaggle/kaggle.json


import kagglehub
path3 = kagglehub.dataset_download("leonardocaravaggio/ge-images4")
path4 = kagglehub.dataset_download("leonardocaravaggio/ge-images6")

100%|██████████| 3.21G/3.21G [00:42<00:00, 80.8MB/s]

Extracting files...


100%|██████████| 1.57G/1.57G [00:14<00:00, 115MB/s] 

Extracting files...


In [3]:
import pandas as pd
ciudades=pd.read_csv("Gini OECD.csv", encoding='latin', sep=';')

ciudades['Gini'] = ciudades['Gini'].str.replace(',', '.', regex=False).astype(float)

In [4]:
import os
import shutil

# Ruta de origen y destino
src_dir = path3+"/Imagenes3"
dst_dir = path3

# Crear lista de archivos
archivos = os.listdir(src_dir)

# Mover sin sobrescribir
for archivo in archivos:
    origen = os.path.join(src_dir, archivo)
    destino = os.path.join(dst_dir, archivo)

    if not os.path.exists(destino):  # si no existe en destino, mover
        shutil.move(origen, destino)
    else:
        print(f"⚠️ Ya existe: {archivo}, no se movió.")

print("✅ Movimiento completado.")

⚠️ Ya existe: Windsor - 5K.png, no se movió.
⚠️ Ya existe: Nantes - 1K.png, no se movió.
⚠️ Ya existe: Gothenburg - 10K.png, no se movió.
⚠️ Ya existe: Gothenburg - 1K.png, no se movió.
⚠️ Ya existe: Saint-Etienne - 5K.png, no se movió.
⚠️ Ya existe: Windsor - 10K.png, no se movió.
⚠️ Ya existe: Calgary - 1K.png, no se movió.
⚠️ Ya existe: Brevard - 15K.png, no se movió.
⚠️ Ya existe: Gothenburg - 5K.png, no se movió.
⚠️ Ya existe: Rennes - 1K.png, no se movió.
⚠️ Ya existe: Albany - 5K.png, no se movió.
⚠️ Ya existe: Gothenburg - 15K.png, no se movió.
⚠️ Ya existe: Dane - 10K.png, no se movió.
⚠️ Ya existe: Calgary - 5K.png, no se movió.
⚠️ Ya existe: Albany - 1K.png, no se movió.
⚠️ Ya existe: Brevard - 10K.png, no se movió.
⚠️ Ya existe: Nantes - 5K.png, no se movió.
⚠️ Ya existe: Brevard - 1K.png, no se movió.
⚠️ Ya existe: Brevard - 5K.png, no se movió.
⚠️ Ya existe: Windsor - 15K.png, no se movió.
⚠️ Ya existe: Dane - 1K.png, no se movió.
⚠️ Ya existe: Windsor - 1K.png, no se mov

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import torch.nn.functional as F
import os
from tqdm import tqdm
from io import BytesIO
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
import pandas as pd

# ---------- FUNCIONES AUXILIARES ----------
def pil_to_bytes(img):
    buf = BytesIO()
    img.save(buf, format='PNG')
    buf.seek(0)
    return buf

def crop_to_square_center(img, size=1773):
    width, height = img.size
    side = min(width, height, size)
    left = (width - side) // 2
    top = (height - side) // 2
    right = left + side
    bottom = top + side
    return img.crop((left, top, right, bottom))

# Transformaciones de imagen
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def build_model(layer_cut):
    """Crea MobileNetV2 cortada hasta la capa indicada."""
    base_model = models.mobilenet_v2(pretrained=True)
    return torch.nn.Sequential(*list(base_model.features[:layer_cut]))

def extract_index(image_filelike, model):
    """Extrae índice de desigualdad (std de avg_pool) de una imagen."""
    image = Image.open(image_filelike).convert("RGB")
    image_tensor = transform(image).unsqueeze(0)  # [1, 3, 224, 224]
    with torch.no_grad():
        features = model(image_tensor)
        avg_pool = F.adaptive_avg_pool2d(features, (1, 1)).squeeze()  # [C]
        inequality_index = np.std(avg_pool.cpu().numpy())
    return inequality_index

# ---------- PREPARAR IMÁGENES ----------
imagenes_5k = []
imagenes_10k = []
ciudades_validas = []

for i in range(len(ciudades)):
    nombre_aglomerado = ciudades.loc[i, "Ciudad"]
    img_5k_path = os.path.join(path3, f"{nombre_aglomerado} - 5K.png")
    img_10k_path = os.path.join(path3, f"{nombre_aglomerado} - 10K.png")
    try:
        img_5k = crop_to_square_center(Image.open(img_5k_path))
        img_10k = crop_to_square_center(Image.open(img_10k_path))
        imagenes_5k.append(pil_to_bytes(img_5k))
        imagenes_10k.append(pil_to_bytes(img_10k))
        ciudades_validas.append(i)
    except Exception as e:
        print(f"⚠️ Error en {nombre_aglomerado}: {e}")
        imagenes_5k.append(None)
        imagenes_10k.append(None)
        ciudades_validas.append(i)

# ---------- LOOP POR CORTES ----------
results = []

# Partición train/test
idx = np.array(ciudades_validas)
train_idx, test_idx = train_test_split(idx, test_size=0.5, random_state=42)

for layer_cut in range(1, 21):
    print(f"\n=== Corte {layer_cut} ===")
    model = build_model(layer_cut)
    model.eval()

    desigualdad_5k = []
    desigualdad_10k = []

    for i in ciudades_validas:
        i_rel = ciudades_validas.index(i)  # índice relativo en listas de imágenes
        img5 = imagenes_5k[i_rel]
        img10 = imagenes_10k[i_rel]
        if img5 is None or img10 is None:
            desigualdad_5k.append(np.nan)
            desigualdad_10k.append(np.nan)
            continue
        try:
            val5 = extract_index(img5, model)
            val10 = extract_index(img10, model)
        except Exception as e:
            print(f"⚠️ Error extrayendo índices en corte {layer_cut}, ciudad {i}: {e}")
            val5, val10 = np.nan, np.nan
        desigualdad_5k.append(val5)
        desigualdad_10k.append(val10)

    # Guardar en DataFrame temporal
    ciudades.loc[ciudades_validas, f"Desigualdad_5km_{layer_cut}"] = desigualdad_5k
    ciudades.loc[ciudades_validas, f"Desigualdad_10km_{layer_cut}"] = desigualdad_10k

    # Calcular correlaciones
    y = ciudades.loc[ciudades_validas, "Gini"]

    # --- 5 km ---
    x5 = ciudades.loc[ciudades_validas, f"Desigualdad_5km_{layer_cut}"]
    mask5 = x5.notna() & y.notna()
    train_mask5 = mask5 & ciudades.index.isin(train_idx)
    test_mask5 = mask5 & ciudades.index.isin(test_idx)
    if train_mask5.sum() > 2 and test_mask5.sum() > 2:
        r_train5, p_train5 = pearsonr(x5[train_mask5], y[train_mask5])
        r_test5, p_test5 = pearsonr(x5[test_mask5], y[test_mask5])
    else:
        r_train5 = p_train5 = r_test5 = p_test5 = np.nan

    # ⬅️ NUEVO: Pearson combinado 5km
    if mask5.sum() > 2:
        r_all5, p_all5 = pearsonr(x5[mask5], y[mask5])
    else:
        r_all5 = p_all5 = np.nan

    # --- 10 km ---
    x10 = ciudades.loc[ciudades_validas, f"Desigualdad_10km_{layer_cut}"]
    mask10 = x10.notna() & y.notna()
    train_mask10 = mask10 & ciudades.index.isin(train_idx)
    test_mask10 = mask10 & ciudades.index.isin(test_idx)
    if train_mask10.sum() > 2 and test_mask10.sum() > 2:
        r_train10, p_train10 = pearsonr(x10[train_mask10], y[train_mask10])
        r_test10, p_test10 = pearsonr(x10[test_mask10], y[test_mask10])
    else:
        r_train10 = p_train10 = r_test10 = p_test10 = np.nan

    # ⬅️ NUEVO: Pearson combinado 10km
    if mask10.sum() > 2:
        r_all10, p_all10 = pearsonr(x10[mask10], y[mask10])
    else:
        r_all10 = p_all10 = np.nan

    results.append({
        "layer_cut": layer_cut,
        "r_train_5km": r_train5, "p_train_5km": p_train5,
        "r_test_5km": r_test5, "p_test_5km": p_test5,
        "r_all_5km": r_all5, "p_all_5km": p_all5,       # ⬅️ NUEVO
        "r_train_10km": r_train10, "p_train_10km": p_train10,
        "r_test_10km": r_test10, "p_test_10km": p_test10,
        "r_all_10km": r_all10, "p_all_10km": p_all10    # ⬅️ NUEVO
    })

In [6]:
# ---------- RESULTADOS ----------
df_results = pd.DataFrame(results)
print("\nResumen de correlaciones por corte:")
df_results


Resumen de correlaciones por corte:


,layer_cut,r_train_5km,p_train_5km,r_test_5km,p_test_5km,r_all_5km,p_all_5km,r_train_10km,p_train_10km,r_test_10km,p_test_10km,r_all_10km,p_all_10km
0,1,0.013286,0.924775,-0.055312,0.694043,-0.024425,0.803731,0.025418,0.856633,-0.060182,0.668597,-0.016621,8.657159e-01
1,2,0.371314,0.006193,0.333720,0.014604,0.365506,0.000117,0.502485,0.000126,0.444215,0.000862,0.487473,1.160851e-07
2,3,0.056107,0.689863,0.053437,0.703926,0.057725,0.556697,0.251886,0.068836,0.248141,0.073205,0.259178,7.301518e-03
3,4,0.147178,0.292953,0.123498,0.378306,0.138214,0.157680,0.277816,0.043994,0.287603,0.036780,0.288821,2.676806e-03
4,5,0.143413,0.305611,0.202871,0.145143,0.172093,0.077735,0.305108,0.026316,0.346213,0.011103,0.330864,5.318760e-04
5,6,0.280387,0.041995,0.383348,0.004606,0.333344,0.000480,0.412342,0.002154,0.486947,0.000218,0.454350,9.971623e-07
6,7,0.229881,0.097740,0.435288,0.001124,0.327607,0.000608,0.386725,0.004230,0.530958,0.000043,0.459946,7.043375e-07
7,8,0.229636,0.098107,0.337886,0.013344,0.285170,0.003047,0.322216,0.018624,0.421642,0.001664,0.376266,7.035992e-05
8,9,0.231684,0.095065,0.426737,0.001440,0.325060,0.000674,0.306847,0.025429,0.492025,0.000183,0.395874,2.664719e-05
9,10,0.290254,0.035004,0.356495,0.008789,0.324862,0.000680,0.332552,0.014975,0.474039,0.000336,0.398414,2.339094e-05
